# AIC 2026 Round 3 Retrieval Evaluation

This notebook evaluates the running Multimodal Retrieval backend against `data/benchmarks/aic_2026_round3.csv` and writes reproducible experiment artifacts for the technical report.

Metrics: Recall@1, Recall@5, Recall@10, Recall@20, and Recall@50. KIS succeeds when a ranked result has the correct video and frame (within `FRAME_TOLERANCE`). QA reports both retrieval recall and strict answer-aware recall. TRAKE succeeds only when one ranked sequence matches the correct video and every ordered ground-truth event frame.

## Protocol

- Rows with `Status != OK` are retained in `per_query_results.csv` but excluded from denominators.
- The API is called at most once per query. Returned results are reused for every k.
- `FRAME_TOLERANCE = 0` is exact-frame evaluation. Change it only if the benchmark protocol permits a tolerance; the value is persisted in run metadata.
- QA retrieval recall ignores answer text; `Strict Recall@k` additionally requires normalized predicted and ground-truth answers to match.
- TRAKE requires one result to match all ordered frame positions. Event-level coverage is logged for diagnosis but is not substituted for strict Recall@k.

In [1]:
from __future__ import annotations

import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests

# Optional: set this when the notebook is launched outside the repository.
PROJECT_ROOT_OVERRIDE: str | None = None

def find_repo_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents]
    if PROJECT_ROOT_OVERRIDE:
        candidates.insert(0, Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve())
    for candidate in candidates:
        if (candidate / 'data' / 'benchmarks' / 'aic_2026_round3.csv').is_file():
            return candidate
    raise RuntimeError(
        'Cannot locate the repository. Set PROJECT_ROOT_OVERRIDE to the folder containing data/benchmarks/aic_2026_round3.csv.'
    )

REPO_ROOT = find_repo_root(Path.cwd())

BENCHMARK_CSV = REPO_ROOT / 'data' / 'benchmarks' / 'aic_2026_round3.csv'
API_BASE = 'http://127.0.0.1:8000'  # Change if the backend is exposed on another host/port.
DATASET_ID: str | None = None        # Optional UUID override; None selects the first READY dataset.

def resolve_dataset_id() -> str:
    if DATASET_ID:
        return DATASET_ID
    response = requests.get(f"{API_BASE.rstrip('/')}/api/datasets", timeout=15)
    response.raise_for_status()
    ready = [item for item in response.json().get('datasets', []) if item.get('status') == 'READY']
    if not ready:
        raise RuntimeError('No READY dataset returned by /api/datasets. Set DATASET_ID to a valid dataset UUID.')
    return str(ready[0]['id'])

DATASET_ID = resolve_dataset_id()
print(f'Using dataset: {DATASET_ID}')
PROFILE = 'competition_default'
TOP_K = 50                           # Must be at least max(RECALL_KS).
RECALL_KS = [1, 5, 10, 20, 50]
FRAME_TOLERANCE = 0                  # Exact benchmark frame matching by default.
REQUEST_TIMEOUT_S = 180
RETRY_COUNT = 2
SLEEP_BETWEEN_QUERIES_S = 0.0

# Optional: run only a small, reproducible smoke-test subset. Set to None for the full benchmark.
QUERY_IDS: list[str] | None = [
    'query-p2-1-kis', 'query-p2-4-kis', 'query-p2-5-qa', 'query-p2-6-qa', 'query-p2-21-trake',
]

# Experiment switches. Keep these values in the report's experimental setup.
USE_QUERY_EXPANSION = True
USE_AGENT_QUERY_PLANNING = True
AGENT_MODEL = 'gpt-4o'               # gpt-4o | gpt-5-nano | gpt-5.6-luna
USE_METADATA = True
USE_RERANKER = True
VISUAL_SEARCH_MODE = 'openclip'      # openclip | siglip2 | both
TEMPORAL_STRATEGY = 'vortex_k_context'
DELTA_T_MAX_MS = 180_000

RUN_ID = datetime.now(timezone.utc).strftime('aic2026_round3_%Y%m%dT%H%M%SZ')
OUTPUT_DIR = REPO_ROOT / 'data' / 'experiments' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
print(f'Artifacts will be written to: {OUTPUT_DIR}')

Using dataset: 917efbbe-44b8-4476-98ef-3e7ec7ca7958
Artifacts will be written to: D:\University\Projects\Individual projects\Multimodal-Retrieval\data\experiments\aic2026_round3_20260911T170234Z


In [2]:
required_columns = {
    'Query ID', 'Query Text', 'Task Type', 'GT Video ID', 'GT Frame ID(s)', 'GT Answer', 'Status'
}
benchmark = pd.read_csv(BENCHMARK_CSV, dtype=str, keep_default_na=False, encoding='utf-8-sig')
missing_columns = required_columns.difference(benchmark.columns)
if missing_columns:
    raise ValueError(f'Benchmark schema is missing columns: {sorted(missing_columns)}')

benchmark['Task Type'] = benchmark['Task Type'].str.upper().str.strip()
benchmark['Status'] = benchmark['Status'].str.upper().str.strip()
if QUERY_IDS is not None:
    requested = set(QUERY_IDS)
    unknown = requested.difference(benchmark['Query ID'])
    if unknown:
        raise ValueError(f'Unknown QUERY_IDS: {sorted(unknown)}')
    benchmark = benchmark[benchmark['Query ID'].isin(QUERY_IDS)].copy()

eligible = benchmark[benchmark['Status'].eq('OK')].copy()
print(f'Total rows: {len(benchmark)} | evaluable rows: {len(eligible)}')
display(benchmark.groupby(['Task Type', 'Status'], dropna=False).size().rename('queries').reset_index())
display(eligible[['Query ID', 'Task Type', 'GT Video ID', 'GT Frame ID(s)', 'GT Answer']].head())

Total rows: 5 | evaluable rows: 5


,Task Type,Status,queries
0,KIS,OK,2
1,QA,OK,2
2,TRAKE,OK,1


,Query ID,Task Type,GT Video ID,GT Frame ID(s),GT Answer
0,query-p2-1-kis,KIS,L26_V424,5582,
3,query-p2-4-kis,KIS,L25_V075,2202,
26,query-p2-5-qa,QA,L25_V016,2513,6
27,query-p2-6-qa,QA,L25_V027,12318,Tây Bắc
34,query-p2-21-trake,TRAKE,L26_V156,"3265, 3780, 4786, 6730",


In [3]:
def parse_frame_ids(raw: str) -> list[int]:
    values = re.findall(r'\d+', str(raw or ''))
    return [int(value) for value in values]

def endpoint_for(task_type: str) -> str:
    return {'QA': '/api/retrieval/qa', 'TRAKE': '/api/retrieval/trake'}.get(
        task_type, '/api/retrieval/search'
    )

def build_payload(row: pd.Series) -> dict[str, Any]:
    task_type = row['Task Type']
    return {
        'dataset_id': DATASET_ID,
        'query_name': row['Query ID'],
        'query_type': task_type,
        'query_text': row['Query Text'],
        'top_k': TOP_K,
        'profile': PROFILE,
        'options': {
            'use_query_expansion': USE_QUERY_EXPANSION,
            'use_agent_query_planning': USE_AGENT_QUERY_PLANNING,
            'agent_model': AGENT_MODEL if USE_AGENT_QUERY_PLANNING else None,
            'use_metadata': USE_METADATA,
            'use_reranker': USE_RERANKER,
            'strict_hybrid': False,
            'visual_search_mode': VISUAL_SEARCH_MODE,
            'delta_t_max_ms': DELTA_T_MAX_MS,
            'temporal_mode': task_type == 'KIS' and False,
            'temporal_strategy': TEMPORAL_STRATEGY,
        },
    }

def call_search(row: pd.Series) -> tuple[dict[str, Any], int]:
    url = f"{API_BASE.rstrip('/')}{endpoint_for(row['Task Type'])}"
    payload = build_payload(row)
    last_error: Exception | None = None
    started = time.perf_counter()
    for attempt in range(RETRY_COUNT + 1):
        try:
            body = json.dumps(payload, ensure_ascii=False).encode('utf-8')
            response = requests.post(
                url, data=body, headers={'Content-Type': 'application/json; charset=utf-8'}, timeout=REQUEST_TIMEOUT_S
            )
            if 400 <= response.status_code < 500:
                raise RuntimeError(f'HTTP {response.status_code}: {response.text[:500]}')
            response.raise_for_status()
            return response.json(), round((time.perf_counter() - started) * 1000)
        except requests.RequestException as exc:
            last_error = exc
            if attempt == RETRY_COUNT:
                break
            time.sleep(min(2 ** attempt, 8))
    raise RuntimeError(f'{url}: {last_error}')

def frontend_display_order(results: list[dict[str, Any]], task_type: str) -> list[dict[str, Any]]:
    """Mirror App.tsx diversifyResultsForDisplay before computing a displayed rank."""
    if task_type not in {'KIS', 'QA'}:
        return list(results)
    first_by_video: list[dict[str, Any]] = []
    deferred: list[dict[str, Any]] = []
    seen_videos: set[str] = set()
    for result in results:
        video_key = str(result.get('video_id') or '')
        if video_key in seen_videos:
            deferred.append(result)
        else:
            seen_videos.add(video_key)
            first_by_video.append(result)
    return first_by_video + deferred

def frame_match(predicted: int | None, expected: int) -> bool:
    # AIC26 compares Frame ID as an integer: no temporal tolerance is permitted.
    return predicted is not None and int(predicted) == expected

def score_result(result: dict[str, Any], row: pd.Series) -> dict[str, Any]:
    task_type = row['Task Type']
    correct_video = str(result.get('video_code') or result.get('video_id') or '') == row['GT Video ID']
    expected_frames = parse_frame_ids(row['GT Frame ID(s)'])
    if task_type == 'TRAKE':
        predicted_frames = [item.get('frame_idx') for item in result.get('sequence_frames', [])]
        if not predicted_frames and result.get('frame_idx') is not None:
            predicted_frames = [result.get('frame_idx')]
        event_matches = [
            frame_match(predicted, expected)
            for predicted, expected in zip(predicted_frames, expected_frames, strict=False)
        ]
        event_coverage = sum(event_matches) / len(expected_frames) if expected_frames else 0.0
        format_valid = (
            bool(result.get('video_code'))
            and len(predicted_frames) == len(expected_frames)
            and all(frame is not None for frame in predicted_frames)
            and predicted_frames == sorted(predicted_frames)
        )
        retrieval_hit = format_valid and correct_video and all(event_matches)
        return {
            'correct_video': correct_video,
            'predicted_frames': predicted_frames,
            'event_coverage': event_coverage,
            'retrieval_hit': retrieval_hit,
            'strict_hit': retrieval_hit,
            'answer_match': None,
            'format_valid': format_valid,
        }

    predicted_frame = result.get('frame_idx')
    retrieval_hit = correct_video and bool(expected_frames) and frame_match(predicted_frame, expected_frames[0])
    answer_match = None
    format_valid = bool(result.get('video_code')) and predicted_frame is not None
    if task_type == 'QA':
        predicted_answer = result.get('answer')
        format_valid = format_valid and isinstance(predicted_answer, str) and len(predicted_answer) <= 100
        # AIC26 requires an exact answer string; do not case-fold, trim, or apply semantic matching.
        answer_match = predicted_answer == row['GT Answer']
    official_match = retrieval_hit and (bool(answer_match) if task_type == 'QA' else True)
    return {
        'correct_video': correct_video,
        'predicted_frames': [predicted_frame] if predicted_frame is not None else [],
        'event_coverage': float(retrieval_hit),
        'retrieval_hit': retrieval_hit,
        'strict_hit': official_match,
        'answer_match': answer_match,
        'format_valid': format_valid,
    }

In [4]:
per_query: list[dict[str, Any]] = []
per_prediction: list[dict[str, Any]] = []
raw_response_path = OUTPUT_DIR / 'raw_responses.jsonl'

with raw_response_path.open('w', encoding='utf-8') as raw_handle:
    for position, (_, row) in enumerate(benchmark.iterrows(), start=1):
        base = {
            'run_id': RUN_ID,
            'query_id': row['Query ID'],
            'task_type': row['Task Type'],
            'query_text': row['Query Text'],
            'status': row['Status'],
            'gt_video_id': row['GT Video ID'],
            'gt_frame_ids': row['GT Frame ID(s)'],
            'gt_answer': row['GT Answer'],
            'evaluable': row['Status'] == 'OK',
        }
        if row['Status'] != 'OK':
            per_query.append({**base, 'api_ok': False, 'latency_ms': None, 'returned_results': 0, 'error': 'Excluded: benchmark status is not OK'})
            continue

        try:
            response, latency_ms = call_search(row)
            results = response.get('results') or []
            displayed_results = frontend_display_order(results, row['Task Type'])
            raw_handle.write(json.dumps({'query_id': row['Query ID'], 'response': response}, ensure_ascii=False) + '\n')
            raw_handle.flush()  # Preserve completed API responses if a long run is interrupted.
            records = []
            for rank, result in enumerate(displayed_results, start=1):
                scored = score_result(result, row)
                record = {
                    **base,
                    'rank': rank,
                    'raw_rank': next((index for index, item in enumerate(results, start=1) if item.get('id') == result.get('id')), rank),
                    'result_id': result.get('id'),
                    'video_code': result.get('video_code'),
                    'frame_idx': result.get('frame_idx'),
                    'frame_indices': json.dumps(scored['predicted_frames']),
                    'answer': result.get('answer'),
                    'score': result.get('score'),
                    'correct_video': scored['correct_video'],
                    'event_coverage': scored['event_coverage'],
                    'retrieval_hit': scored['retrieval_hit'],
                    'strict_hit': scored['strict_hit'],
                    'answer_match': scored['answer_match'],
                    'official_format_valid': scored['format_valid'],
                    'official_exact_match': scored['strict_hit'],
                }
                records.append(record)
                per_prediction.append(record)

            query_record = {**base, 'api_ok': True, 'latency_ms': latency_ms, 'returned_results': len(results), 'error': ''}
            for k in RECALL_KS:
                top_k_records = records[:k]
                query_record[f'official_hit_at_{k}'] = any(item['strict_hit'] for item in top_k_records)
            query_record['best_event_coverage_at_50'] = max((item['event_coverage'] for item in records[:50]), default=0.0)
            per_query.append(query_record)
            print(f'[{position:02d}/{len(benchmark):02d}] {row["Query ID"]}: {len(results)} results, {latency_ms} ms')
        except Exception as exc:
            per_query.append({**base, 'api_ok': False, 'latency_ms': None, 'returned_results': 0, 'error': str(exc)})
            print(f'[{position:02d}/{len(benchmark):02d}] {row["Query ID"]}: ERROR {exc}')
        if SLEEP_BETWEEN_QUERIES_S:
            time.sleep(SLEEP_BETWEEN_QUERIES_S)

per_query_df = pd.DataFrame(per_query)
per_prediction_df = pd.DataFrame(per_prediction)
per_query_df.to_csv(OUTPUT_DIR / 'per_query_results.csv', index=False, encoding='utf-8-sig')
per_prediction_df.to_csv(OUTPUT_DIR / 'per_prediction_results.csv', index=False, encoding='utf-8-sig')
display(per_query_df[['query_id', 'task_type', 'api_ok', 'returned_results', 'latency_ms', 'error']])

[01/05] query-p2-1-kis: 50 results, 11018 ms


[02/05] query-p2-4-kis: 50 results, 9569 ms


[03/05] query-p2-5-qa: 50 results, 10904 ms


[04/05] query-p2-6-qa: 50 results, 8009 ms


[05/05] query-p2-21-trake: 50 results, 15548 ms


,query_id,task_type,api_ok,returned_results,latency_ms,error
0,query-p2-1-kis,KIS,True,50,11018,
1,query-p2-4-kis,KIS,True,50,9569,
2,query-p2-5-qa,QA,True,50,10904,
3,query-p2-6-qa,QA,True,50,8009,
4,query-p2-21-trake,TRAKE,True,50,15548,


In [5]:
def metric_rows(frame: pd.DataFrame, slice_name: str) -> list[dict[str, Any]]:
    evaluated = frame[frame['evaluable'] & frame['api_ok']].copy()
    rows: list[dict[str, Any]] = []
    for k in RECALL_KS:
        rows.append({
            'run_id': RUN_ID,
            'slice': slice_name,
            'metric': f'Official Recall@{k}',
            'value': evaluated[f'official_hit_at_{k}'].astype(float).mean() if len(evaluated) else float('nan'),
            'numerator': int(evaluated[f'official_hit_at_{k}'].sum()) if len(evaluated) else 0,
            'denominator': len(evaluated),
        })
    return rows

metric_records = metric_rows(per_query_df, 'overall')
for task_type, task_frame in per_query_df.groupby('task_type', dropna=False):
    metric_records.extend(metric_rows(task_frame, f'task={task_type}'))

metrics_df = pd.DataFrame(metric_records)
metrics_df.to_csv(OUTPUT_DIR / 'metrics_summary.csv', index=False, encoding='utf-8-sig')
failures_df = per_query_df[per_query_df['evaluable'] & (~per_query_df['api_ok'] | per_query_df['official_hit_at_50'].eq(False))]
failures_df.to_csv(OUTPUT_DIR / 'failures.csv', index=False, encoding='utf-8-sig')

run_config = {
    'run_id': RUN_ID, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'benchmark_csv': str(BENCHMARK_CSV), 'api_base': API_BASE, 'dataset_id': DATASET_ID,
    'profile': PROFILE, 'top_k': TOP_K, 'recall_ks': RECALL_KS,
    'frame_tolerance': FRAME_TOLERANCE, 'agent_model': AGENT_MODEL,
    'use_query_expansion': USE_QUERY_EXPANSION, 'use_agent_query_planning': USE_AGENT_QUERY_PLANNING,
    'use_metadata': USE_METADATA, 'use_reranker': USE_RERANKER,
    'visual_search_mode': VISUAL_SEARCH_MODE, 'temporal_strategy': TEMPORAL_STRATEGY,
    'delta_t_max_ms': DELTA_T_MAX_MS,
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding='utf-8')

display(metrics_df.pivot(index='metric', columns='slice', values='value').style.format('{:.3f}'))
print('Wrote:')
for path in sorted(OUTPUT_DIR.iterdir()):
    print(' -', path.name)

slice,overall,task=KIS,task=QA,task=TRAKE
metric,,,,
Official Recall@1,0.000,0.000,0.000,0.000
Official Recall@10,0.000,0.000,0.000,0.000
Official Recall@20,0.000,0.000,0.000,0.000
Official Recall@5,0.000,0.000,0.000,0.000
Official Recall@50,0.000,0.000,0.000,0.000


Wrote:
 - failures.csv
 - metrics_summary.csv
 - per_prediction_results.csv
 - per_query_results.csv
 - raw_responses.jsonl
 - run_config.json


## Report-ready interpretation

Use `metrics_summary.csv` for aggregate tables and `per_query_results.csv` plus `per_prediction_results.csv` for error analysis. State the exact backend profile, model selection, visual mode, temporal strategy, top-k, and frame tolerance from `run_config.json`. Do not compare runs with different denominator rules or frame tolerances.